In [1]:
!pip install langchain langgraph langchain-google-genai langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.5/558.5 kB 26.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9


In [7]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver

In [8]:
from google.colab import userdata
groq_api_key = userdata.get("GROQ_API_KEY")

In [9]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=groq_api_key,
)

In [11]:
class JokeState(TypedDict):
  topic: str
  joke: str
  explaination: str

In [12]:
def generate_joke(state: JokeState):
  topic = state['topic']
  prompt = f'Generate a joke on the topic {topic}'
  response = llm.invoke(prompt).content
  return { 'joke': response }

In [13]:
def generate_joke_explaination(state: JokeState):
  joke = state['joke']
  prompt = f'Write an explaination for the joke - {joke}'
  response = llm.invoke(prompt).content
  return { 'explaination': response }

In [23]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_joke_explaination', generate_joke_explaination)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_joke_explaination')
graph.add_edge('generate_joke_explaination', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [37]:
config2 = {"configurable": {"thread_id": "1"}}
workflow.invoke({"topic": "Football"}, config=config2)

{'topic': 'Football',
 'joke': 'Why did the football go to the doctor?\n\nBecause it was feeling a little "deflated"! (get it?)',
 'explaination': 'A classic play on words. This joke is funny because it uses a clever pun to create a humorous connection between the setup and the punchline.\n\nIn this joke, the setup is "Why did the football go to the doctor?" which primes the listener to expect a reason related to an injury or illness. The punchline "Because it was feeling a little \'deflated\'!" is a clever play on words because "deflated" has a double meaning here.\n\nIn a literal sense, a football can become deflated if it loses air, which would affect its ability to be used in a game. However, the word "deflated" also has a figurative meaning, which is to feel unenthusiastic, depressed, or lacking energy. So, the joke is saying that the football went to the doctor because it was feeling unwell, but the reason for its illness is a clever wordplay on the fact that footballs can also b

In [35]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football go to the doctor?\n\nBecause it was feeling a little "deflated"! (get it?)', 'explaination': 'A classic play on words. The joke "Why did the football go to the doctor? Because it was feeling a little \'deflated\'!" is a clever pun that uses a double meaning of the word "deflated" to create humor.\n\nIn one sense, a football can become deflated when it loses air, which is a common issue with inflatable balls. However, the word "deflated" can also be used to describe someone\'s emotional state, where they feel disheartened, depressed, or lacking energy.\n\nThe joke relies on this dual meaning, setting up the expectation that the football has gone to the doctor for a physical issue (perhaps a puncture or a leak). But instead, the punchline subverts this expectation by using the word "deflated" to imply that the football is feeling down or unwell in an emotional sense. The added phrase "get it?" is a nod to the wordpl

In [38]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football go to the doctor?\n\nBecause it was feeling a little "deflated"! (get it?)', 'explaination': 'A classic play on words. This joke is funny because it uses a clever pun to create a humorous connection between the setup and the punchline.\n\nIn this joke, the setup is "Why did the football go to the doctor?" which primes the listener to expect a reason related to an injury or illness. The punchline "Because it was feeling a little \'deflated\'!" is a clever play on words because "deflated" has a double meaning here.\n\nIn a literal sense, a football can become deflated if it loses air, which would affect its ability to be used in a game. However, the word "deflated" also has a figurative meaning, which is to feel unenthusiastic, depressed, or lacking energy. So, the joke is saying that the football went to the doctor because it was feeling unwell, but the reason for its illness is a clever wordplay on the fact that 